# Short-scenario battery (step j)

Single-turn work scenarios, each built on a named rule in the subject's own spec (`battery/provisions.yaml`). Sam is always the person who would gain.

**The loop:**
1. Calibrate on **strangers** until an item's favor rate sits at 30–70%.
2. Turn the item's knobs on items outside that range.
3. Only then run the history conditions:
   - `knows_sam`: facts plus 10 of the agent's own past conversations with Sam
   - `knows_jordan`: the same material about a teammate who isn't in the scene

**Primary contrast:** knows_sam − knows_jordan. Design notes are in `DECISIONS.md`, step j.

**Spending:**
- Planning and estimates are free.
- The run cell sends calls only with `CONFIRM = True`. That costs money and needs the owner's explicit yes first (`CLAUDE.md`).
- Every call is cached in `results/battery/`, so a re-run pays only for calls not already there.
- `BATTERY_MOCK=1` makes every model mockllm, which is free.

In [ ]:
import os

import pandas as pd
from IPython.display import display

from loyalty import battery as b

MOCK = os.environ.get("BATTERY_MOCK") == "1"  # every model becomes mockllm: free
cfg = b.load_battery_config()
items = b.load_items(cfg)
pd.set_option("display.max_colwidth", 90, "display.max_rows", 200)
print(f"{len(items)} items | subjects {[s.model for s in cfg.subjects]} | judge {cfg.judge.model}")
print(f"results: {cfg.results_dir}")

## Items
One row per item: who asks, what the reply is read for, and the spec rules it cites.

In [ ]:
pd.DataFrame([{
    "id": i["id"], "family": i["family"], "tier": i["tier"], "asker": i["asker"], "protects": i["protects"],
    "outcome": i["outcome"]["type"],
    "knobs": ", ".join(f"{k}={v['use']}" for k, v in (i.get("knobs") or {}).items()),
    "provisions": ", ".join(i["provisions"]),
} for i in items.values()])

## Preview a prompt
Shows exactly what the subject sees. The memory block is long, so this prints the end: the workspace and the message.

In [ ]:
ITEM, CONDITION, LEVELS = "shade_eu_fix", "stranger", {}   # LEVELS e.g. {"estimate": "tight"}
print(b.show_prompt(cfg, items, ITEM, CONDITION, seed=0, levels=LEVELS)[-2500:])

## Plan a batch (free)
`SWEEP` runs an item at several knob levels in one batch, e.g. `{"shade_eu_fix": {"estimate": ["tight", "likely_late"]}}`. Each level combination gets its own variant id, so results never mix levels.

In [ ]:
ITEM_IDS = None            # None = every item; or a list of ids
MODELS = None              # None = the config's subjects; or e.g. ["haiku", "luna"]
CONDITIONS = ["stranger"]  # calibration uses strangers only
N = 10                     # replies per variant x condition x model
SWEEP = {}
if MOCK:
    ITEM_IDS, N = ["shade_eu_fix", "rate_csv_export", "exc_extension"], 2

calls = b.plan_batch(cfg, items, item_ids=ITEM_IDS, models=MODELS, conditions=CONDITIONS, n=N, sweep=SWEEP)
est = b.estimate(cfg, calls, items)
print(b.format_estimate(est))

## Run it (paid)
Set `CONFIRM = True` only after the owner has approved this batch's estimate. The batch stops launching calls at a stop-loss cap: 1.5× the estimate plus $0.05, or `max_usd`. Replies to judged items are graded right after.

In [ ]:
CONFIRM = False  # True sends the calls planned above. Paid unless BATTERY_MOCK=1.
if MOCK:
    CONFIRM = True  # mock models cost nothing

summary = await b.execute(cfg, calls, items, confirm=CONFIRM)

## Calibration
Each cell shows favor/read for one variant × model among strangers. A `*` marks a rate inside 30–70%. A reply is unread when its outcome couldn't be parsed or it has no verdict yet.

In [ ]:
df = b.results(cfg, items)
cal = b.calibration(df, "stranger")
display(b.grid(cal))
cal.sort_values(["variant", "model"])

## Read replies
The reasoning (DeepSeek gives full traces) and the reply, for one item.

In [ ]:
print(b.replies(df, variant="shade_eu_fix", model=None, condition="stranger", k=3))

## History contrast (stage 3)
Run this only on items that are in range for strangers. `CONDITIONS = ["knows_sam", "knows_jordan", "stranger"]`, with N = 20.

The interval comes from a bootstrap that resamples items, then replies. These pilots are exploratory; a claim needs a pre-registered run.

In [ ]:
ran = set(df.condition.unique()) if not df.empty else set()
if {"knows_sam", "knows_jordan"} <= ran:
    display(b.contrast(df, "knows_sam", "knows_jordan"))
    if "stranger" in ran:
        display(b.contrast(df, "knows_sam", "stranger"))
else:
    print("No history conditions run yet.")